In [ ]:
# ── 1. Setup & Data Loading ──────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math, glob

# Load combined dataset (merged from 12 monthly CSVs)
all_trips = pd.read_csv("/Users/nithish/Downloads/google capstone/combined_divvy_tripdata_2025.csv")

print(f"Rows   : {len(all_trips):,}")
print(f"Columns: {len(all_trips.columns)}")
print(f"\nColumns: {all_trips.columns.tolist()}")


In [ ]:
# ── 2. Data Cleaning ─────────────────────────────────────────────────────────

# Rename columns for clarity
all_trips.rename(columns={
    'rideable_type': 'bike_type',
    'member_casual': 'user_type'
}, inplace=True)

# Convert timestamps
all_trips['started_at'] = pd.to_datetime(all_trips['started_at'])
all_trips['ended_at']   = pd.to_datetime(all_trips['ended_at'])

# Ride duration in minutes
all_trips['ride_length_m'] = (
    (all_trips['ended_at'] - all_trips['started_at']).dt.total_seconds() / 60
)

# Derived time columns
all_trips['day_of_week'] = all_trips['started_at'].dt.dayofweek   # 0=Mon, 6=Sun
all_trips['day_name']    = all_trips['started_at'].dt.day_name()
all_trips['month_num']   = all_trips['started_at'].dt.month
all_trips['month']       = all_trips['started_at'].dt.strftime('%B')
all_trips['hour']        = all_trips['started_at'].dt.hour

# Season mapping
season_map = {
    12:'Winter', 1:'Winter', 2:'Winter',
     3:'Spring', 4:'Spring', 5:'Spring',
     6:'Summer', 7:'Summer', 8:'Summer',
     9:'Fall',  10:'Fall',  11:'Fall'
}
all_trips['season'] = all_trips['month_num'].map(season_map)

# Remove invalid rides (negative or zero duration, > 24 hrs)
before = len(all_trips)
all_trips = all_trips[
    (all_trips['ride_length_m'] > 0) &
    (all_trips['ride_length_m'] <= 1440)
].reset_index(drop=True)

print(f"Rows removed (invalid duration): {before - len(all_trips):,}")
print(f"Clean dataset: {len(all_trips):,} rows")
print(f"\nColumns: {all_trips.columns.tolist()}")


In [ ]:
# ── 3. Dataset Overview ──────────────────────────────────────────────────────
print("Ride share by user type:")
print(all_trips['user_type'].value_counts().to_string())

print("\nBike type breakdown:")
print(all_trips['bike_type'].value_counts().to_string())

print("\nMissing values:")
print(all_trips.isnull().sum()[all_trips.isnull().sum() > 0].to_string())

all_trips.head()


In [ ]:
# ── 4. Summary Statistics ────────────────────────────────────────────────────
summary = all_trips.groupby('user_type')['ride_length_m'].agg(
    Mean='mean', Median='median', Max='max', Min='min'
).round(2)
print("Ride Length Summary (minutes):")
print(summary.to_string())

print(f"\nMode day of week: {all_trips['day_name'].mode()[0]}")
print(f"Mode month      : {all_trips['month'].mode()[0]}")


In [ ]:
# ── 5a. Pivot: Avg Ride Length by User Type x Day of Week ───────────────────
day_labels  = {0:'Mon', 1:'Tue', 2:'Wed', 3:'Thu', 4:'Fri', 5:'Sat', 6:'Sun'}
ordered_days = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']

def mins_to_mmss(m):
    if isinstance(m, float) and math.isnan(m): return '--:--'
    s = int(round(m * 60))
    return f"{s//60:02d}:{s%60:02d}"

pivot_avg = (
    all_trips.groupby(['user_type','day_of_week'])['ride_length_m']
    .mean().unstack('day_of_week').rename(columns=day_labels)
    .reindex(columns=ordered_days)
)

try:    pivot_fmt = pivot_avg.map(mins_to_mmss)
except: pivot_fmt = pivot_avg.applymap(mins_to_mmss)

print("Average Ride Length (MM:SS) by User Type x Day of Week")
print("=" * 58)
print(pivot_fmt.to_string())


In [ ]:
# ── 5b. Pivot: Ride Count by User Type x Day of Week ────────────────────────
pivot_count = (
    all_trips.groupby(['user_type','day_of_week'])['ride_id']
    .count().unstack('day_of_week').rename(columns=day_labels)
    .reindex(columns=ordered_days)
)
print("Number of Rides by User Type x Day of Week")
print("=" * 58)
print(pivot_count.to_string())


In [ ]:
# ── 6. Season Analysis ───────────────────────────────────────────────────────
season_order = ['Winter','Spring','Summer','Fall']

# Ride counts
pivot_season_count = (
    all_trips.groupby(['user_type','season'])['ride_id']
    .count().unstack('season').reindex(columns=season_order)
)
print("Ride Count by User Type x Season")
print("=" * 45)
print(pivot_season_count.to_string())

# Avg ride length
pivot_season_avg = (
    all_trips.groupby(['user_type','season'])['ride_length_m']
    .mean().unstack('season').reindex(columns=season_order).round(2)
)
print("\nAvg Ride Length (min) by User Type x Season")
print("=" * 45)
print(pivot_season_avg.to_string())


In [ ]:
# ── 7. Viz 1: Ride Share Split (Casual vs Member) ────────────────────────────
ride_share = all_trips['user_type'].value_counts()
colors     = ['#4C9BE8','#F4845F']

fig, ax = plt.subplots(figsize=(6,6))
wedges, texts, autotexts = ax.pie(
    ride_share, labels=ride_share.index.str.capitalize(),
    autopct='%1.1f%%', colors=colors, startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=2),
    textprops=dict(fontsize=13)
)
for at in autotexts: at.set_fontsize(13); at.set_fontweight('bold')
ax.set_title('Ride Share: Casual vs Member (2025)', fontsize=15, fontweight='bold', pad=20)
plt.tight_layout(); plt.show()


In [ ]:
# ── 7. Viz 2: Number of Rides by Day of Week ────────────────────────────────
rides_by_day = (
    all_trips.groupby(['user_type','day_of_week'])['ride_id'].count()
    .reset_index()
)
rides_by_day['day_name'] = rides_by_day['day_of_week'].map(day_labels)
rides_by_day['day_name'] = pd.Categorical(rides_by_day['day_name'], categories=ordered_days, ordered=True)
rides_by_day = rides_by_day.sort_values('day_name')

fig, ax = plt.subplots(figsize=(10,5))
sns.barplot(data=rides_by_day, x='day_name', y='ride_id', hue='user_type',
            palette={'casual':'#F4845F','member':'#4C9BE8'},
            edgecolor='white', linewidth=1.2, ax=ax)
ax.set_title('Number of Rides by Day of Week', fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Day of Week', fontsize=12); ax.set_ylabel('Number of Rides', fontsize=12)
ax.legend(title='User Type', fontsize=11, title_fontsize=11)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{int(x):,}'))
ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.show()


In [ ]:
# ── 7. Viz 3: Avg Ride Length by Day of Week ────────────────────────────────
avg_len_day = (
    all_trips.groupby(['user_type','day_of_week'])['ride_length_m'].mean()
    .reset_index()
)
avg_len_day['day_name'] = avg_len_day['day_of_week'].map(day_labels)
avg_len_day['day_name'] = pd.Categorical(avg_len_day['day_name'], categories=ordered_days, ordered=True)
avg_len_day = avg_len_day.sort_values('day_name')

fig, ax = plt.subplots(figsize=(10,5))
sns.barplot(data=avg_len_day, x='day_name', y='ride_length_m', hue='user_type',
            palette={'casual':'#F4845F','member':'#4C9BE8'},
            edgecolor='white', linewidth=1.2, ax=ax)
for c in ax.containers: ax.bar_label(c, fmt='%.1f min', fontsize=9, padding=3)
ax.set_title('Average Ride Length by Day of Week', fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Day of Week', fontsize=12); ax.set_ylabel('Avg Ride Length (min)', fontsize=12)
ax.legend(title='User Type', fontsize=11, title_fontsize=11)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.show()


In [ ]:
# ── 7. Viz 4: Number of Rides by Season ─────────────────────────────────────
rides_by_season = (
    all_trips.groupby(['user_type','season'])['ride_id'].count().reset_index()
)
rides_by_season['season'] = pd.Categorical(rides_by_season['season'], categories=season_order, ordered=True)

fig, ax = plt.subplots(figsize=(9,5))
sns.barplot(data=rides_by_season, x='season', y='ride_id', hue='user_type',
            palette={'casual':'#F4845F','member':'#4C9BE8'},
            edgecolor='white', linewidth=1.2, ax=ax)
for c in ax.containers: ax.bar_label(c, fmt='%,.0f', fontsize=9, padding=3)
ax.set_title('Number of Rides by Season', fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Season', fontsize=12); ax.set_ylabel('Number of Rides', fontsize=12)
ax.legend(title='User Type', fontsize=11, title_fontsize=11)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{int(x):,}'))
ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.show()


In [ ]:
# ── 7. Viz 5: Avg Ride Length by Season ─────────────────────────────────────
avg_len_season = (
    all_trips.groupby(['user_type','season'])['ride_length_m'].mean().reset_index()
)
avg_len_season['season'] = pd.Categorical(avg_len_season['season'], categories=season_order, ordered=True)

fig, ax = plt.subplots(figsize=(9,5))
sns.barplot(data=avg_len_season, x='season', y='ride_length_m', hue='user_type',
            palette={'casual':'#F4845F','member':'#4C9BE8'},
            edgecolor='white', linewidth=1.2, ax=ax)
for c in ax.containers: ax.bar_label(c, fmt='%.1f min', fontsize=9, padding=3)
ax.set_title('Average Ride Length by Season', fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Season', fontsize=12); ax.set_ylabel('Avg Ride Length (min)', fontsize=12)
ax.legend(title='User Type', fontsize=11, title_fontsize=11)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.show()


In [ ]:
# ── 7. Viz 6: Monthly Ride Trend ─────────────────────────────────────────────
month_names  = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
                7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}

rides_by_month = (
    all_trips.groupby(['user_type','month_num'])['ride_id'].count().reset_index()
)
rides_by_month['month_name'] = rides_by_month['month_num'].map(month_names)
rides_by_month['month_name'] = pd.Categorical(
    rides_by_month['month_name'], categories=list(month_names.values()), ordered=True)

fig, ax = plt.subplots(figsize=(11,5))
for user, color, marker in [('casual','#F4845F','o'),('member','#4C9BE8','s')]:
    d = rides_by_month[rides_by_month['user_type']==user].sort_values('month_num')
    ax.plot(d['month_name'], d['ride_id'], label=user.capitalize(),
            color=color, marker=marker, linewidth=2.5, markersize=7)
    ax.fill_between(d['month_name'], d['ride_id'], alpha=0.1, color=color)
ax.set_title('Monthly Ride Trend: Casual vs Member (2025)', fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Month', fontsize=12); ax.set_ylabel('Number of Rides', fontsize=12)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{int(x):,}'))
ax.legend(title='User Type', fontsize=11, title_fontsize=11)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.show()


In [ ]:
# ── 7. Viz 7: Rides by Hour of Day ───────────────────────────────────────────
rides_by_hour = (
    all_trips.groupby(['user_type','hour'])['ride_id'].count().reset_index()
)
fig, ax = plt.subplots(figsize=(12,5))
for user, color, marker in [('casual','#F4845F','o'),('member','#4C9BE8','s')]:
    d = rides_by_hour[rides_by_hour['user_type']==user].sort_values('hour')
    ax.plot(d['hour'], d['ride_id'], label=user.capitalize(),
            color=color, marker=marker, linewidth=2.5, markersize=6)
    ax.fill_between(d['hour'], d['ride_id'], alpha=0.1, color=color)
ax.axvspan(7,  9,  alpha=0.08, color='gray',  label='Morning commute (7–9am)')
ax.axvspan(16, 19, alpha=0.08, color='green', label='Evening commute (4–7pm)')
ax.set_title('Rides by Hour of Day: Casual vs Member', fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Hour of Day', fontsize=12); ax.set_ylabel('Number of Rides', fontsize=12)
ax.set_xticks(range(0,24))
ax.set_xticklabels([f'{h:02d}:00' for h in range(24)], rotation=45, fontsize=8)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{int(x):,}'))
ax.legend(fontsize=10)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.show()


In [ ]:
# ── 7. Viz 8: Rides by Bike Type ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14,5))

# Count
rides_by_bike = (
    all_trips.groupby(['user_type','bike_type'])['ride_id'].count()
    .reset_index().rename(columns={'ride_id':'ride_count'})
)
sns.barplot(data=rides_by_bike, x='bike_type', y='ride_count', hue='user_type',
            palette={'casual':'#F4845F','member':'#4C9BE8'},
            edgecolor='white', linewidth=1.2, ax=axes[0])
for c in axes[0].containers: axes[0].bar_label(c, fmt='%,.0f', fontsize=9, padding=3)
axes[0].set_title('Number of Rides by Bike Type', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Bike Type'); axes[0].set_ylabel('Number of Rides')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{int(x):,}'))
axes[0].legend(title='User Type'); axes[0].spines[['top','right']].set_visible(False)

# Avg duration
avg_by_bike = (
    all_trips.groupby(['user_type','bike_type'])['ride_length_m'].mean()
    .reset_index().rename(columns={'ride_length_m':'avg_minutes'})
)
sns.barplot(data=avg_by_bike, x='user_type', y='avg_minutes', hue='bike_type',
            palette={'classic_bike':'#4C9BE8','electric_bike':'#F4845F'},
            edgecolor='white', linewidth=1.2, ax=axes[1])
for c in axes[1].containers: axes[1].bar_label(c, fmt='%.1f min', fontsize=9, padding=3)
axes[1].set_title('Avg Ride Length by Bike Type', fontsize=13, fontweight='bold')
axes[1].set_xlabel('User Type'); axes[1].set_ylabel('Avg Ride Length (min)')
axes[1].set_xticklabels(['Casual','Member']); axes[1].legend(title='Bike Type')
axes[1].spines[['top','right']].set_visible(False)

plt.suptitle('Bike Type Analysis: Casual vs Member', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()


In [ ]:
# ── 8. Key Findings ──────────────────────────────────────────────────────────
print("""
╔══════════════════════════════════════════════════════════════╗
║              KEY FINDINGS — CYCLISTIC 2025                  ║
╚══════════════════════════════════════════════════════════════╝

1. RIDE VOLUME
   • Members account for the majority of total rides
   • Casuals take fewer but significantly longer rides

2. DAY OF WEEK BEHAVIOR
   • Members peak on weekdays (Mon–Fri) → commuter pattern
   • Casuals peak on weekends (Sat–Sun) → leisure pattern
   • Casual rides are longer than member rides on every single day

3. TIME OF DAY
   • Members spike at 8am and 5pm → work commute
   • Casuals build gradually, peaking 2pm–5pm → recreational use

4. SEASONAL TRENDS
   • Both groups peak in Summer, drop in Winter
   • Casuals drop far more steeply in Winter → fair-weather riders
   • Even in Winter, casuals still ride longer than members on average

5. BIKE TYPE
   • Casuals spend more time on classic bikes (leisure, sightseeing)
   • Electric bikes used for shorter trips across both groups
""")


In [ ]:
# ── 9. Top 3 Recommendations ─────────────────────────────────────────────────
print("""
╔══════════════════════════════════════════════════════════════╗
║         TOP 3 RECOMMENDATIONS FOR CYCLISTIC MARKETING       ║
╚══════════════════════════════════════════════════════════════╝

RECOMMENDATION 1: Launch a Weekend Membership Tier
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Casuals ride most on weekends and for longer durations.
A discounted weekend-only membership directly matches
their existing behavior and lowers the conversion barrier.
→ Target : Casual riders active on Sat–Sun
→ Message: "You already ride on weekends — make it cheaper."

RECOMMENDATION 2: Summer Conversion Campaign
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Casual ridership peaks in Summer (Jun–Aug). This is the
highest-intent window to convert. A limited-time Summer
membership offer launched in May captures riders before
they hit peak usage.
→ Target : Casual riders in Spring / early Summer
→ Message: "Ride all Summer for less — join before June."

RECOMMENDATION 3: Promote Commuter Benefits to Casual Riders
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Casuals show no 8am/5pm spike — they don't commute yet.
Showing them cost-per-ride savings and health benefits
can expand usage from weekend leisure to weekday commutes,
making membership far more valuable to them.
→ Target : Casual riders near high-traffic commuter stations
→ Message: "Save money. Skip traffic. Ride to work."
""")
